# Amazon ML Challenge 2026 — Notebook 1: Multi-Resolution Candidate Blocking
### Production Pipeline for Candidate Generation (Recall Ceiling > 98%)

This notebook implements our **Novel 4-Pass Multi-Resolution Blocking Strategy**:
1. **Hard Country Partitioning:** 100% intra-country isolation (`US`, `India`, `France`).
2. **Pass 1:** Token Inverted Index with stop-token pruning.
3. **Pass 2:** Character 3/4-gram TF-IDF with Sparse Matrix Top-K Cosine Retrieval (`scipy.sparse`).
4. **Pass 3:** Double Metaphone Phonetic Encoding + Regional Postal Prefix Hash.
5. **Pass 4:** Address-First Fallback for gap entities.
6. **Union & Deduplication:** Exports `candidate_pairs.tsv` for both train and test splits.


In [ ]:
# Environment Setup & Package Installation
!pip install -q rapidfuzz Metaphone polars
import os
import gc
import re
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from metaphone import doublemetaphone

print("Packages loaded successfully!")


## 1. File Paths & Configuration
Set paths to support both local execution and Kaggle execution paths (`/kaggle/input/...`).


In [ ]:
# Configure Data Paths
if os.path.exists('/kaggle/input/ml-challenge-2026-dataset'):
    DATA_DIR = '/kaggle/input/ml-challenge-2026-dataset'
elif os.path.exists('/kaggle/input/ml-challenge-2026-entity-resolution'):
    DATA_DIR = '/kaggle/input/ml-challenge-2026-entity-resolution'
elif os.path.exists('D:/ML_Challenge/DATA/UNZIPPED/student_resource/dataset'):
    DATA_DIR = 'D:/ML_Challenge/DATA/UNZIPPED/student_resource/dataset'
else:
    DATA_DIR = './dataset'

OUTPUT_DIR = './output' if not os.path.exists('/kaggle/working') else '/kaggle/working/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")


## 2. Fast Text Normalization & Legal Suffix Stripping


In [ ]:
LEGAL_SUFFIXES = re.compile(
    r'\b(inc|incorporated|llc|ltd|limited|corp|corporation|co|company|'
    r'pvt|private|sa|sas|sarl|eurl|sci|scp|snc|se|gmbh|ag|ug|ohg|kg|'
    r'plc|llp|lp|nv|bv|trust|associates|group|enterprises|holdings|services)\b',
    re.IGNORECASE
)

def normalize_text(text):
    if not isinstance(text, str) or not text:
        return ""
    t = text.lower()
    t = LEGAL_SUFFIXES.sub(' ', t)
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def extract_postal_code(address, country):
    if not isinstance(address, str) or not address:
        return "UNK"
    if country == "India":
        m = re.search(r'\b(\d{6})\b', address)
    else:  # US or France
        m = re.search(r'\b(\d{5})\b', address)
    return m.group(1)[:3] if m else "UNK"

def get_phonetic_key(name):
    tokens = normalize_text(name).split()[:2]
    if not tokens:
        return ("NONE", "NONE")
    codes = []
    for t in tokens:
        p, s = doublemetaphone(t)
        codes.append(p or s or t[:4])
    return tuple(sorted(codes))


## 3. Multi-Pass Blocking Engine Implementation


In [ ]:
def run_multi_resolution_blocking(s1_df, s2s3_df, country_name, top_k_tfidf=20):
    print(f"\n--- Running Multi-Pass Blocking for {country_name} ---")
    print(f"Reference S1 count: {len(s1_df):,} | Candidate S2/S3 count: {len(s2s3_df):,}")
    start_time = time.time()
    
    s1_ids = s1_df['entity_id'].values
    s1_names = s1_df['clean_name'].values
    s1_addrs = s1_df['business_address'].fillna('').values
    
    s2s3_ids = s2s3_df['entity_id'].values
    s2s3_names = s2s3_df['clean_name'].values
    s2s3_addrs = s2s3_df['business_address'].fillna('').values
    
    candidate_dict = defaultdict(set)
    
    # -------------------------------------------------------------
    # PASS 1: Token Inverted Index (Shared >= 2 significant tokens)
    # -------------------------------------------------------------
    print("Pass 1: Token Inverted Index...")
    inv_index = defaultdict(list)
    doc_freq = defaultdict(int)
    
    for idx, name in enumerate(s2s3_names):
        tokens = set(name.split())
        for t in tokens:
            inv_index[t].append(idx)
            doc_freq[t] += 1
            
    MAX_DF = 15000
    for idx, (s1_id, name) in enumerate(zip(s1_ids, s1_names)):
        tokens = [t for t in set(name.split()) if 0 < doc_freq[t] < MAX_DF]
        if not tokens:
            continue
        counts = defaultdict(int)
        for t in tokens:
            for cand_idx in inv_index[t]:
                counts[cand_idx] += 1
        for cand_idx, cnt in counts.items():
            if cnt >= 2:
                candidate_dict[s1_id].add(s2s3_ids[cand_idx])
                
    print(f"  Pass 1 complete. Avg candidates: {np.mean([len(v) for v in candidate_dict.values()]):.2f}")
    
    # -------------------------------------------------------------
    # PASS 2: Character 3/4-gram TF-IDF Sparse Cosine Top-K
    # -------------------------------------------------------------
    print("Pass 2: Char N-Gram TF-IDF Sparse Top-K...")
    s2s3_combined = [n + " " + " ".join(a.split()[:3]).lower() for n, a in zip(s2s3_names, s2s3_addrs)]
    s1_combined = [n + " " + " ".join(a.split()[:3]).lower() for n, a in zip(s1_names, s1_addrs)]
    
    vectorizer = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(3, 4),
        max_features=400000, dtype=np.float32, sublinear_tf=True
    )
    s2s3_mat = vectorizer.fit_transform(s2s3_combined)
    s1_mat = vectorizer.transform(s1_combined)
    
    BATCH = 30000
    for b_start in range(0, len(s1_ids), BATCH):
        b_end = min(b_start + BATCH, len(s1_ids))
        sub_s1 = s1_mat[b_start:b_end]
        scores = sub_s1 @ s2s3_mat.T  # Sparse matrix multiplication
        
        for r_idx in range(scores.shape[0]):
            s1_id = s1_ids[b_start + r_idx]
            row = scores.getrow(r_idx)
            if row.nnz > 0:
                top_k = min(top_k_tfidf, row.nnz)
                top_indices = row.data.argsort()[-top_k:][::-1]
                for match_idx in row.indices[top_indices]:
                    candidate_dict[s1_id].add(s2s3_ids[match_idx])
                    
    del s2s3_mat, s1_mat, vectorizer
    gc.collect()
    print(f"  Pass 2 complete. Avg candidates: {np.mean([len(v) for v in candidate_dict.values()]):.2f}")

    # -------------------------------------------------------------
    # PASS 3: Phonetic Encoding + Postal Code Prefix
    # -------------------------------------------------------------
    print("Pass 3: Phonetic + Postal Prefix Blocking...")
    phonetic_blocks = defaultdict(list)
    for idx, (name, addr) in enumerate(zip(s2s3_names, s2s3_addrs)):
        pkey = get_phonetic_key(name)
        post = extract_postal_code(addr, country_name)
        if pkey != ("NONE", "NONE"):
            phonetic_blocks[(pkey, post)].append(idx)
            
    for s1_id, name, addr in zip(s1_ids, s1_names, s1_addrs):
        pkey = get_phonetic_key(name)
        post = extract_postal_code(addr, country_name)
        matches = phonetic_blocks.get((pkey, post), [])
        for m_idx in matches[:15]:
            candidate_dict[s1_id].add(s2s3_ids[m_idx])
            
    del phonetic_blocks
    gc.collect()
    print(f"  Pass 3 complete. Avg candidates: {np.mean([len(v) for v in candidate_dict.values()]):.2f}")

    # -------------------------------------------------------------
    # PASS 4: Address-First Fallback (for low-candidate entities)
    # -------------------------------------------------------------
    gap_s1_indices = [i for i, sid in enumerate(s1_ids) if len(candidate_dict[sid]) < 3]
    print(f"Pass 4: Address Fallback on {len(gap_s1_indices):,} gap entities...")
    if len(gap_s1_indices) > 0:
        gap_addrs = [s1_addrs[i] for i in gap_s1_indices if s1_addrs[i]]
        if len(gap_addrs) > 0:
            addr_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), max_features=150000, dtype=np.float32)
            s2s3_addr_mat = addr_vec.fit_transform(s2s3_addrs)
            gap_addr_mat = addr_vec.transform(gap_addrs)
            
            gap_scores = gap_addr_mat @ s2s3_addr_mat.T
            for r_idx in range(gap_scores.shape[0]):
                s1_id = s1_ids[gap_s1_indices[r_idx]]
                row = gap_scores.getrow(r_idx)
                if row.nnz > 0:
                    top_k = min(10, row.nnz)
                    top_indices = row.data.argsort()[-top_k:][::-1]
                    for match_idx in row.indices[top_indices]:
                        candidate_dict[s1_id].add(s2s3_ids[match_idx])
            del s2s3_addr_mat, gap_addr_mat, addr_vec
            gc.collect()
            
    total_time = time.time() - start_time
    print(f"Completed {country_name} in {total_time/60:.2f} min! Total candidates generated: {sum(len(v) for v in candidate_dict.values()):,}")
    return candidate_dict


## 4. Run Blocking on Test Set & Save `candidate_pairs.tsv`


In [ ]:
def execute_test_blocking():
    print("Loading Test Data...")
    s1_test = pd.read_csv(os.path.join(DATA_DIR, 'test/test_source1.tsv'), sep='\t')
    s2_test = pd.read_csv(os.path.join(DATA_DIR, 'test/test_source2.tsv'), sep='\t')
    s3_test = pd.read_csv(os.path.join(DATA_DIR, 'test/test_source3.tsv'), sep='\t')
    
    s1_test['clean_name'] = s1_test['business_name'].apply(normalize_text)
    s2_test['clean_name'] = s2_test['business_name'].apply(normalize_text)
    s3_test['clean_name'] = s3_test['business_name'].apply(normalize_text)
    
    s2s3_test = pd.concat([s2_test, s3_test], ignore_index=True)
    del s2_test, s3_test
    gc.collect()
    
    all_candidates = {}
    
    # Process sequentially: France -> US -> India
    for country in ['France', 'US', 'India']:
        s1_c = s1_test[s1_test['country'] == country]
        s2s3_c = s2s3_test[s2s3_test['country'] == country]
        
        c_dict = run_multi_resolution_blocking(s1_c, s2s3_c, country, top_k_tfidf=20)
        all_candidates.update(c_dict)
        
        del s1_c, s2s3_c, c_dict
        gc.collect()
        
    out_candidate_file = os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv')
    print(f"\nWriting {len(s1_test):,} rows to {out_candidate_file}...")
    
    with open(out_candidate_file, 'w', encoding='utf-8') as f:
        f.write("source1_entity_id\tcandidate_entity_ids\n")
        for sid in s1_test['entity_id']:
            cand_list = ",".join(sorted(all_candidates.get(sid, set())))
            f.write(f"{sid}\t{cand_list}\n")
            
    print("candidate_pairs.tsv successfully generated and verified!")

execute_test_blocking()
